# Part 1 - Forecast reanalysis -> coarsened target 125 m wind

Forecast the coarsened target 125 m wind (u, v) at leads +1 / +7 / +14 days from reanalysis fields. Train on 2016-2020; validate on a held-out tail (autumn 2020).

- Tier 0 - persistence.
- Tier 1 - per-target LightGBM.

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import reanalysis_loader, splits
import forecast_features as ff
import forecast_model as fm

In [ ]:
all_dates = [d for d in reanalysis_loader.list_dates() if splits.target_available(d)]
issue = all_dates[::5]
LEADS = (1, 7, 14)
df = ff.build_training_table(issue, hours=(0, 12), leads_days=LEADS)
print("rows:", len(df), "| issue dates:", df['time'].dt.date.nunique())

In [ ]:
cut = pd.Timestamp("2020-09-01")   # hold out the tail of the last train year
tr, va = df[df.time < cut], df[df.time >= cut]
print("train rows:", len(tr), "| val rows:", len(va))

In [ ]:
models = fm.train_lgbm(tr, leads_days=LEADS)
pred = fm.predict_lgbm(models, va)
pu, pv = fm.persistence_forecast(va)
rows = []
for lead in LEADS:
    yu, yv = va[f"u125c_d{lead}"], va[f"v125c_d{lead}"]
    rows.append({
        "lead_days": lead,
        "RMSE_persistence": fm.uv_rmse(yu, yv, pu, pv),
        "RMSE_lgbm":        fm.uv_rmse(yu, yv, pred[f"u125c_d{lead}"], pred[f"v125c_d{lead}"]),
        "cMAE_persistence_deg": fm.circular_mae_from_uv(yu, yv, pu, pv),
        "cMAE_lgbm_deg":    fm.circular_mae_from_uv(yu, yv, pred[f"u125c_d{lead}"], pred[f"v125c_d{lead}"]),
    })
results = pd.DataFrame(rows)
results

## Results

In [ ]:
fm.save_models(models, "models")
import os
saved = sorted(f for f in os.listdir("models") if f.endswith(".joblib"))
print("saved", len(saved), "models to ./models/:")
for f in saved:
    print(" ", f)